# Level 2 — Analytics & Pattern Discovery
### Business Analytics Demo

---

## What is Level 2 Analytics?

**Level 2 Analytics** is your data science co-pilot. It transforms natural language questions into SQL queries, performs customer segmentation using machine learning, analyzes sentiment from social media, and generates actionable insights.

| What you can ask | What it does |
|---|---|
| "Segment customers by risk and engagement" | K-means clustering with visualizations |
| "What's the average balance for high-value customers?" | SQL analytics with aggregations |
| "Analyze sentiment from customer feedback" | RoBERTa-based sentiment analysis |
| "Summarize this support ticket" | BART text summarization |
| "Show me Customer 360 view for CUST-005" | Unified view across CRM, sales, social, support |

**No coding required.** Ask in plain English — Level 2 does the analysis.

---

## How it works

```
Your question
     │
     ▼
1. Intent Parsing — classify the question (aggregation, segmentation, correlation)
     │
     ▼
2. Schema Mapping — map natural-language terms to database columns
     │
     ▼
3. Query Generation — build safe SQL via LangChain create_sql_query_chain
     │
     ▼
4. Execution & Retrieval — run read-only query (SQL_MAX_ROWS limit)
     │
     ▼
5. Post-Processing — compute statistics, generate charts (data/charts/)
     │
     ▼
6. Guardrail — redact PII and forbidden phrases before returning
     │
     ▼
Returns insights + charts
```

Every query is logged to the audit trail with full traceability.

---

## Setup

Run this cell once to initialize the system.

In [1]:
import sys, os
from pathlib import Path

# Resolve project root
_root = Path(os.getcwd())
while not (_root / "nonagentic" / "__init__.py").exists() and _root != _root.parent:
    _root = _root.parent
_notebooks = _root / "notebooks"
os.chdir(_root)
sys.path.insert(0, str(_root))
sys.path.insert(0, str(_notebooks))

from dotenv import load_dotenv
load_dotenv()

# Import all necessary modules
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, HTML, Markdown, Image
import json

# Import Level 2 tools
from nonagentic.tools.analytics import run_sql_query, generate_segment
from nonagentic.tools.customer360 import get_customer_360, get_sales_analytics, get_sentiment_analytics, get_support_analytics
from nonagentic.tools.nlp import analyze_sentiment, summarize_text, analyze_batch_sentiment
from nonagentic.tools.visualisation import visualise

# Import utilities
from notebooks.nonagentic.utils import *

print("✅ Level 2 Analytics Ready")
print(f"📁 Project root: {_root}")


✅ Level 2 Analytics Ready
📁 Project root: /home/kahloka/projects/agenticAI


---
## Use Case 1 — Customer Segmentation (K-means Clustering)

**Business scenario**: The marketing team wants to identify distinct customer groups based on risk and engagement scores to tailor campaigns.

K-means clustering automatically discovers customer segments. No manual rules needed — the algorithm finds natural groupings in the data.

In [2]:
# Perform K-means segmentation
result = generate_segment(algorithm="kmeans", n_clusters=4)
segments = result.get("segments", [])

# Use utility function to display results
display_segmentation_results(segments)


Segment,Size,Avg Risk Score,Avg Engagement
cluster_1,57,0.132,0.736
cluster_0,49,0.173,0.397
cluster_3,35,0.341,0.430
cluster_2,67,0.304,0.158


---
## Use Case 2 — SQL Analytics (Top Customers by Balance)

**Business scenario**: The account manager needs to identify top customers by lifetime value for VIP treatment.

SQL queries are generated and executed automatically from natural language via `create_sql_query_chain`. A safety guard rejects any mutating statements.

In [3]:
# Query top customers by lifetime value
queries = get_test_queries()
result = run_sql_query(queries["top_customers_sql"])

# Use utility function to display results
display_sql_analytics_results(result, title="💰 Top 10 Customers by Lifetime Value")


Customer ID,Name,Segment,Lifetime Value,Fraud Score,Engagement
CUST-004,Sharon Kennedy,at_risk,"€89,205.38",0.300,0.440
CUST-104,الأستاذ ثروت خندف,dormant_vip,"€88,661.71",0.570,0.200
CUST-149,Michelle Wallace,dormant_vip,"€88,343.24",0.570,0.240
CUST-037,صدّام جرهم,new,"€87,845.00",0.220,0.590
CUST-165,Gary Morales,vip,"€87,761.18",0.300,0.900
CUST-045,Lukácsné Kovács Beát,new,"€87,456.71",0.870,0.680
CUST-073,Sheri Johnson,dormant_vip,"€87,217.77",0.060,0.070
CUST-067,Dr. Kiss Ildikó,casual,"€86,056.98",0.050,0.570
CUST-131,لاما عكاوي,dormant_vip,"€86,006.77",0.720,0.180
CUST-048,Veres Mária,new,"€85,543.75",0.590,0.430


---
## Use Case 3 — Fraud Fraud Risk Analysis (High-Fraud-Risk Customers)

**Business scenario**: Trust & Safety needs to identify high-fraud-risk customers for enhanced verification.

Customers are filtered by fraud score and their characteristics analyzed via read-only SQL.

In [4]:
# Query high-fraud-risk customers
sql = """
SELECT segment, COUNT(*) as count, AVG(fraud_score) as avg_risk, AVG(engagement_score) as avg_engagement
FROM customers
WHERE fraud_score > 0.7
GROUP BY segment
ORDER BY count DESC
"""

result = run_sql_query(sql)

# Use utility function to display results
display_fraud_risk_analysis(result)


segment,count,avg_risk,avg_engagement
at_risk,17,0.838,0.448
dormant_vip,16,0.814,0.143
casual,11,0.822,0.494
new,10,0.797,0.491
vip,9,0.890,0.821


---
## Use Case 4 — Sentiment Analysis (Social Media Feedback)

**Business scenario**: The brand team wants to understand customer sentiment from social media posts.

**RoBERTa** (`cardiffnlp/twitter-roberta-base-sentiment-latest`) classifies each post as positive, neutral, or negative. Batch variant available via `analyze_batch_sentiment`.

In [5]:
# Load social media data
with open("data/social_media.json") as f:
    social_data = json.load(f)

# Analyze sentiment for sample posts
sample_posts = social_data[:10]

# Analyze each post
results = []
for post in sample_posts:
    sentiment_result = analyze_sentiment(post["text"])
    results.append({
        "Text": post["text"][:60] + "..." if len(post["text"]) > 60 else post["text"],
        "Platform": post["platform"],
        "Sentiment": sentiment_result.get("sentiment", "unknown"),
        "Confidence": f"{sentiment_result.get('confidence', 0):.1%}"
    })

# Use enhanced visualization function
display_enhanced_sentiment_analysis(results, sample_posts=sample_posts)


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.pooler.dense.bias       | UNEXPECTED |  | 
roberta.pooler.dense.weight     | UNEXPECTED |  | 
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Text,Platform,Sentiment,Confidence
Been a loyal customer for years. Never disappointed!,facebook,positive,96.7%
Return process is unnecessarily complicated. Avoid!,twitter,negative,84.5%
Poor quality for the price. Expected much better,instagram,negative,91.2%
Lightning fast response from customer service team,facebook,positive,68.6%
Website keeps crashing during checkout. Lost my cart multipl...,facebook,negative,92.3%
Product categories are well organized on the site,reddit,positive,84.1%
Lightning fast response from customer service team,facebook,positive,68.6%
They have a good variety of brands available,instagram,positive,88.7%
Do they ship internationally? Need to send a gift abroad,instagram,neutral,87.0%
Smooth checkout process and arrived perfectly packaged,instagram,positive,95.4%


---
## Use Case 5 — Text Summarization (Support Tickets)

**Business scenario**: Support managers need quick summaries of lengthy support tickets.

**BART** (`facebook/bart-large-cnn`) generates concise summaries. Returns `compression_ratio`. Skips texts shorter than `min_length` words.

In [6]:
# Load support tickets
with open("data/support_tickets.json") as f:
    support_data = json.load(f)

# Get a sample ticket
sample_ticket = support_data[0]

# Use utility function to display summarization
display_text_summarization(sample_ticket, summarize_text)


---
## Use Case 6 — Customer 360 View (Unified Data)

**Business scenario**: A account manager needs a complete view of a customer before a meeting.

Data from **CRM, sales, social media, and support** is aggregated into one unified view with summary metrics.

In [7]:
# Get Customer 360 view
demo_customers = get_demo_customers()
customer_id = demo_customers["customer_360_demo"]
customer_360 = get_customer_360(customer_id)

# Use utility function to display comprehensive view
display_customer_360_view(customer_360, customer_id)


Category
home
books
clothing


Channel,Consent
marketing,True
email,True
sms,True
phone,False


---
## Use Case 7 — Sales Analytics (Revenue by Category)

**Business scenario**: The sales team wants to understand which categories generate the most revenue.

Sales transactions are aggregated by `product_category` and `channel`. Returns `total_revenue`, `by_product`, `by_channel`, and `top_product`.

In [8]:
# Get sales analytics
sales_analytics = get_sales_analytics()

# Use utility function to display dashboard
display_sales_analytics_dashboard(sales_analytics)


---
## Use Case 8 — Support Analytics (Ticket Analysis)

**Business scenario**: The support manager needs to understand ticket volume, types, and resolution times.

Support tickets are aggregated by type, status, and priority. Returns `avg_resolution_hours` and `high_priority` count.

In [9]:
# Get support analytics
support_analytics = get_support_analytics()

# Use utility function to display dashboard
display_support_analytics_dashboard(support_analytics)


---
## Summary

| Use Case | What it does | Technology |
|---|---|---|
| **1. Customer Segmentation** | K-means clustering by risk & engagement | scikit-learn |
| **2. SQL Analytics** | Text-to-SQL via `create_sql_query_chain` | LangChain + SQLAlchemy |
| **3. Fraud Risk Analysis** | Filter high-fraud-risk customers | SQL aggregations |
| **4. Sentiment Analysis** | Classify social media sentiment | RoBERTa (transformers) |
| **5. Text Summarization** | Summarize support tickets | BART (transformers) |
| **6. Customer 360 View** | Unified view across all data sources | Multi-source integration |
| **7. Sales Analytics** | Revenue by category and channel | Pandas aggregations |
| **8. Support Analytics** | Ticket volume and resolution metrics | Pandas aggregations |

---

### Key Business Benefits

- **Data-Driven Decisions** — Discover patterns and insights automatically
- **No SQL Required** — Ask questions in plain English
- **ML-Powered** — K-means clustering, sentiment analysis, text summarization
- **Unified View** — Customer 360 across CRM, sales, social, support
- **Read-Only Safety** — SQL injection guard rejects mutating statements
- **Guardrail** — PII and forbidden phrases redacted before output
- **Visualizations** — Charts saved to `data/charts/` automatically
- **Audit Trail** — Every query logged for trust & safety

---

### What Comes Next

Level 2 is the **analytical** layer — it only reads and analyses data, never modifies it. When you need to *act* on insights:

- **Level 3** — Execute actions (send emails, create cases, update records)
- **Level 4** — Strategic campaigns (multi-step workflows, A/B testing, KPI tracking)

---

### Technical Notes

**Models Used**:
- K-means clustering (scikit-learn)
- RoBERTa for sentiment analysis (cardiffnlp/twitter-roberta-base-sentiment-latest)
- BART for summarization (facebook/bart-large-cnn)

**Data Sources**:
- `data/customers.db` — Customer profiles (SQLite)
- `data/sales_transactions.json` — Sales data
- `data/social_media.json` — Social media posts
- `data/support_tickets.json` — Support tickets

**Charts Saved To**: `data/charts/`

**Audit Log**: `data/audit.jsonl`